# 01 - Data Ingestion and Schema Review

**Purpose:** load the raw credit-risk workbook, standardize sheet/column names, validate schema, preserve the correct row grain, and save the first merged interim dataset.

This notebook owns only **ingestion and schema review**. It does not perform cleaning, EDA, statistical testing, feature engineering, or modelling.

## Stage ownership

| Stage | This notebook does | This notebook does not do |
|---|---|---|
| Raw workbook loading | Yes | No model preparation |
| Sheet and column standardization | Yes | No imputation |
| Expected-column validation | Yes | No one-hot/ordinal encoding |
| Duplicate borrower-ID review | Yes | No statistical testing |
| Safe merge using `user_id + record_sequence` | Yes | No feature selection |
| Interim dataset save | Yes | No cleaning decisions beyond merge integrity |

In [1]:
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.append(str(SRC_DIR))

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"
REPORT_DIR = PROJECT_ROOT / "reports"
TABLE_DIR = REPORT_DIR / "tables"
FIGURE_DIR = REPORT_DIR / "figures"

for path in [INTERIM_DIR, PROCESSED_DIR, TABLE_DIR, FIGURE_DIR]:
    path.mkdir(parents=True, exist_ok=True)

TARGET_COL = "defaulter"

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


from credit_risk.data.ingestion import (
    read_source_workbook,
    standardize_workbook_sheets,
    normalize_id_columns,
    merge_credit_risk_sheets,
)
from credit_risk.data.schema import summarize_workbook, duplicate_id_summary
from credit_risk.data.validation import validate_expected_columns, validate_target

RAW_PATH = RAW_DIR / "Credit_Risk_Dataset.xlsx"
INTERIM_PATH = INTERIM_DIR / "credit_risk_merged_interim.csv"

RAW_PATH

WindowsPath('D:/Banking and Finance/Projects/canadian-retail-credit-risk-xai/data/raw/Credit_Risk_Dataset.xlsx')

## 1. Load source workbook

In [2]:
if not RAW_PATH.exists():
    raise FileNotFoundError(
        f"Raw workbook not found at {RAW_PATH}. Place Credit_Risk_Dataset.xlsx in data/raw/."
    )

raw_sheets = read_source_workbook(RAW_PATH)

source_sheet_overview = pd.DataFrame([
    {
        "sheet_name": sheet_name,
        "row_count": sheet_df.shape[0],
        "column_count": sheet_df.shape[1],
        "columns": ", ".join(map(str, sheet_df.columns)),
    }
    for sheet_name, sheet_df in raw_sheets.items()
])

source_sheet_overview.to_csv(TABLE_DIR / "01_source_sheet_overview.csv", index=False)
source_sheet_overview

,sheet_name,row_count,column_count,columns
0,loan_information,134417,5,"User_id, Loan Category, Amount, Interest Rate, Tenure(years)"
1,Employment,134417,7,"User id, Employmet type, Tier of Employment, Industry, Role, Work Experience, Total Income(PA)"
2,Personal_information,134417,8,"User id, Gender, Married, Dependents, Home, Pincode, Social Profile, Is_verified"
3,Other_information,134417,7,"User_id, Delinq_2yrs, Total Payement , Received Principal, Interest Received, Number of loans, Defaulter"


## 2. Standardize sheet and column names

In [3]:
sheets = standardize_workbook_sheets(raw_sheets)
sheets = normalize_id_columns(sheets)

standardized_sheet_overview = pd.DataFrame([
    {
        "sheet_name": sheet_name,
        "row_count": sheet_df.shape[0],
        "column_count": sheet_df.shape[1],
        "columns": ", ".join(sheet_df.columns),
    }
    for sheet_name, sheet_df in sheets.items()
])

standardized_sheet_overview.to_csv(TABLE_DIR / "01_standardized_sheet_overview.csv", index=False)
standardized_sheet_overview

,sheet_name,row_count,column_count,columns
0,loan_information,134417,5,"user_id, loan_category, amount, interest_rate, tenure_years"
1,employment,134417,7,"user_id, employment_type, tier_of_employment, industry, role, work_experience, total_income_pa"
2,personal_information,134417,8,"user_id, gender, married, dependents, home, pincode, social_profile, is_verified"
3,other_information,134417,7,"user_id, delinq_2yrs, total_payment, received_principal, interest_received, number_of_loans, defaulter"


## 3. Schema summary and expected-column validation

In [4]:
schema_summary = summarize_workbook(sheets)
expected_column_check = validate_expected_columns(sheets)

schema_summary.to_csv(TABLE_DIR / "01_schema_summary.csv", index=False)
expected_column_check.to_csv(TABLE_DIR / "01_expected_column_check.csv", index=False)

display(Markdown("### Schema summary"))
display(schema_summary)

display(Markdown("### Expected-column validation"))
display(expected_column_check)

### Schema summary

,dataset,column,dtype,non_null_count,missing_count,missing_pct,unique_values
0,loan_information,user_id,int64,134417,0,0.0000,133752
1,loan_information,loan_category,object,134417,0,0.0000,7
2,loan_information,amount,float64,106798,27619,20.5500,86157
3,loan_information,interest_rate,float64,134417,0,0.0000,137
4,loan_information,tenure_years,int64,134417,0,0.0000,2
5,employment,user_id,int64,134417,0,0.0000,133752
6,employment,employment_type,object,54731,79686,59.2800,2
7,employment,tier_of_employment,object,54731,79686,59.2800,7
8,employment,industry,object,134413,4,0.0000,12974
9,employment,role,object,134417,0,0.0000,46


### Expected-column validation

,dataset,expected_column,present
0,loan_information,user_id,True
1,loan_information,loan_category,True
2,loan_information,amount,True
3,loan_information,interest_rate,True
4,loan_information,tenure_years,True
5,employment,user_id,True
6,employment,employment_type,True
7,employment,tier_of_employment,True
8,employment,industry,True
9,employment,role,True


## 4. Duplicate borrower-ID review before merge

In [5]:
duplicate_summary = duplicate_id_summary(sheets)
duplicate_summary.to_csv(TABLE_DIR / "01_duplicate_user_id_summary.csv", index=False)
duplicate_summary

,dataset,id_column_present,row_count,unique_id_count,duplicate_id_count
0,loan_information,True,134417,133752,665
1,employment,True,134417,133752,665
2,personal_information,True,134417,133752,665
3,other_information,True,134417,133752,665


## 5. Safe merge source sheets

The source workbook can contain repeated `user_id` values. Therefore, this project uses the safe observation grain:

> `user_id + record_sequence`

This avoids accidental many-to-many merges and row-count inflation.

In [6]:
expected_row_count = next(iter(sheets.values())).shape[0]

merged_df = merge_credit_risk_sheets(sheets)

required_keys = ["user_id", "record_sequence"]
missing_keys = [col for col in required_keys if col not in merged_df.columns]
if missing_keys:
    raise ValueError(f"Missing required merge key columns after merge: {missing_keys}")

record_key_duplicate_count = int(merged_df[required_keys].duplicated().sum())

merge_validation = pd.DataFrame({
    "check": [
        "expected_row_count_from_first_sheet",
        "merged_row_count",
        "merged_column_count",
        "record_sequence_present",
        "record_key_duplicate_count",
        "row_count_preserved",
    ],
    "result": [
        expected_row_count,
        merged_df.shape[0],
        merged_df.shape[1],
        "record_sequence" in merged_df.columns,
        record_key_duplicate_count,
        merged_df.shape[0] == expected_row_count,
    ],
})

if merged_df.shape[0] != expected_row_count:
    raise ValueError(
        f"Merged row count changed from {expected_row_count:,} to {merged_df.shape[0]:,}."
    )

if record_key_duplicate_count != 0:
    raise ValueError(f"Found {record_key_duplicate_count:,} duplicate user_id + record_sequence keys.")

merge_validation.to_csv(TABLE_DIR / "01_merge_validation.csv", index=False)
merge_validation

,check,result
0,expected_row_count_from_first_sheet,134417
1,merged_row_count,134417
2,merged_column_count,25
3,record_sequence_present,True
4,record_key_duplicate_count,0
5,row_count_preserved,True


## 6. Target validation at ingestion stage

In [7]:
target_check = validate_target(merged_df, target=TARGET_COL)
target_distribution = (
    merged_df[TARGET_COL]
    .value_counts(dropna=False)
    .rename_axis(TARGET_COL)
    .reset_index(name="row_count")
)
target_distribution["row_share_pct"] = (target_distribution["row_count"] / len(merged_df) * 100).round(4)

pd.DataFrame([target_check]).to_csv(TABLE_DIR / "01_target_validation.csv", index=False)
target_distribution.to_csv(TABLE_DIR / "01_target_distribution.csv", index=False)

display(pd.DataFrame([target_check]))
display(target_distribution)

,target_present,row_count,target_missing_count,target_values,default_rate
0,True,134417,0,"[0, 1]",0.0904


,defaulter,row_count,row_share_pct
0,0,122264,90.9587
1,1,12153,9.0413


## 7. Save interim merged dataset

In [8]:
INTERIM_PATH.parent.mkdir(parents=True, exist_ok=True)
merged_df.to_csv(INTERIM_PATH, index=False)

save_summary = pd.DataFrame({
    "output": ["interim_merged_dataset"],
    "path": [str(INTERIM_PATH.relative_to(PROJECT_ROOT))],
    "row_count": [merged_df.shape[0]],
    "column_count": [merged_df.shape[1]],
})

save_summary.to_csv(TABLE_DIR / "01_interim_save_summary.csv", index=False)
save_summary

,output,path,row_count,column_count
0,interim_merged_dataset,data\interim\credit_risk_merged_interim.csv,134417,25


## Carry-forward decisions

1. Use `user_id + record_sequence` as the row-level observation grain.
2. Do not merge later notebooks on `user_id` alone.
3. Use the interim file `data/interim/credit_risk_merged_interim.csv` as the source for Notebook 02.
4. Keep target validation separate from model evaluation.
5. Leave missingness, leakage review, cleaning, EDA, feature engineering, and modelling to later notebooks.